# Formula 1 Tyre Strategy Analysis Using Python 🏎️
### Academic Research & Exploratory Data Analysis Notebook
**Course Code:** CUTM1018 - Data Analytics | Centurion University of Technology and Management
**Authors:** Pratik Raj (250301120218), Swayam Suraj Nayak (250301120227), Debashish Pradhan (250301120238)
**Supervisor:** Sabyasachi Mohanty

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.io as pio

# Ensure project root is accessible
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data_loader import setup_fastf1_cache, load_race_session, extract_session_payload
from src.data_cleaning import clean_race_laps, get_representative_laps
from src.stint_analysis import extract_all_stints, generate_driver_strategy_summary
from src.tyre_analysis import calculate_compound_usage_summary, calculate_compound_stint_metrics, calculate_compound_pace_metrics
from src.degradation_analysis import analyze_race_degradation, compare_compound_degradation
from src.pitstop_analysis import extract_all_pit_stops, calculate_driver_pit_summary, calculate_strategy_frequency
from src.strategy_analysis import build_strategy_position_dataset, analyze_stops_vs_position, calculate_strategy_correlations
from src.visualization import plot_stint_timeline, plot_compound_usage, plot_driver_degradation

print("All analytical modules imported successfully!")

## 1. Data Ingestion & Caching
Loading the 2023 Bahrain Grand Prix session telemetry.

In [ ]:
setup_fastf1_cache()
session = load_race_session(year=2023, gp_identifier="Bahrain", session_type="R")
payload = extract_session_payload(session)
print(f"Event: {payload['metadata']['official_name']}")
print(f"Total Laps: {payload['metadata']['total_laps']} | Drivers: {len(payload['drivers'])}")

## 2. Non-Destructive Data Cleaning Pipeline
Converting Timedeltas to seconds and tagging in-laps, out-laps, and caution periods.

In [ ]:
cleaned_laps, audit_report = clean_race_laps(payload["laps"])
print("Data Cleaning Audit Summary:")
for k, v in audit_report.items():
    print(f"  • {k}: {v}")

## 3. Tyre Stint Detection & Strategy Sequences
Identifying contiguous tyre stints and synthesizing strategy descriptions (e.g. S → S → H).

In [ ]:
stints_df = extract_all_stints(cleaned_laps)
strategy_summary = generate_driver_strategy_summary(stints_df)
strategy_summary.head(8)

## 4. Tyre Degradation Modeling with Fuel Correction
Fitting OLS regression slopes and quantifying compound wear rates.

In [ ]:
degradation_df = analyze_race_degradation(cleaned_laps)
compound_deg = compare_compound_degradation(degradation_df)
compound_deg

## 5. Pit Stop & Strategy Frequency Analysis

In [ ]:
pit_stops_df = extract_all_pit_stops(cleaned_laps)
pit_summary = calculate_driver_pit_summary(pit_stops_df, all_drivers=payload["drivers"], results_df=payload["results"])
strat_freq = calculate_strategy_frequency(pit_summary)
strat_freq

## 6. Strategy vs. Race Classification (Correlation vs. Causation)

In [ ]:
strategy_pos_df = build_strategy_position_dataset(payload["results"], strategy_summary, pit_summary)
corr_audit = calculate_strategy_correlations(strategy_pos_df)
print("Spearman Correlation (Stops vs Finish):", corr_audit["spearman_stops_vs_finish"])
print("Spearman Correlation (Grid vs Finish):", corr_audit["spearman_grid_vs_finish"])
print("\nAcademic Rationale:", corr_audit["academic_note"])

## 7. Interactive Stint Timeline Visualization

In [ ]:
fig_timeline = plot_stint_timeline(stints_df, payload["results"])
fig_timeline.show()